<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Image_Super_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Super-Resolution using ESRGAN
This notebook demonstrates how to use a pre-trained Generative Adversarial Network (ESRGAN) to upscale low-resolution images.

In [1]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
print("TF version:", tf.__version__)
print("Hub version:", hub.__version__)

# Set environment variable to handle some TF Hub caching issues if necessary
os.environ["TFHUB_DOWNLOAD_PROGRESS"] = "True"

TF version: 2.19.0
Hub version: 0.16.1


### Load the ESRGAN Model
We use the ESRGAN model hosted on TensorFlow Hub. This model can upscale images by 4x while maintaining high-frequency details.

In [ ]:
SAVED_MODEL_PATH = "https://tfhub.dev/captain-pool/esrgan-tf2/1"
model = hub.load(SAVED_MODEL_PATH)

Downloaded https://tfhub.dev/captain-pool/esrgan-tf2/1, Total size: 20.60MB



### Helper Functions
We need functions to load images, preprocess them into tensors, and visualize the results.

In [ ]:
import numpy as np
from PIL import Image

def preprocess_image(image_path):
    """ Loads image from path and adds batch dimension. """
    hr_image = tf.image.decode_image(tf.io.read_file(image_path))
    # If PNG, remove the alpha channel. The model expects RGB.
    if hr_image.shape[-1] == 4:
        hr_image = hr_image[..., :-1]
    hr_size = (tf.convert_to_tensor(hr_image.shape[:-1]) // 4) * 4
    hr_image = tf.image.crop_to_bounding_box(hr_image, 0, 0, hr_size[0], hr_size[1])
    hr_image = tf.cast(hr_image, tf.float32)
    return tf.expand_dims(hr_image, 0)

def save_image(image, filename):
    """ Saves unscaled Tensor Images. """
    if not isinstance(image, Image.Image):
        image = tf.clip_by_value(image, 0, 255)
        image = Image.fromarray(tf.cast(image, tf.uint8).numpy())
    image.save(f"{filename}.jpg")
    print(f"Saved as {filename}.jpg")

def plot_comparison(low_res, high_res):
    """ Plots the original vs super-resolution image. """
    plt.figure(figsize=(15, 10))
    plt.subplot(1, 2, 1)
    plt.title("Original (Low Res)")
    plt.imshow(low_res[0] / 255.0)
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.title("ESRGAN (High Res)")
    plt.imshow(high_res[0] / 255.0)
    plt.axis("off")
    plt.show()

### Run Super-Resolution
You can upload your own image or use a sample URL. Here, we'll download a sample low-resolution image to test.

In [ ]:
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/b/b1/Yellow_Labrador_Retriever_pup_700.jpg"
image_path = tf.keras.utils.get_file("sample_image.jpg", IMAGE_URL)

lr_image = preprocess_image(image_path)

# Running the model
fake_image = model(lr_image)
fake_image = tf.squeeze(fake_image)

# Visualization
plot_comparison(lr_image, tf.expand_dims(fake_image, 0))